In [1]:
# RCF + cellulosic ethanol without dilute-acid pretreatment

from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.cellulosic_ethanol_no_preatreatment import create_cellulosic_ethanol_system
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.cellulosic_tea import create_cellulosic_ethanol_tea

from biosteam import main_flowsheet as F
import biosteam as bst

chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7

chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_CRUDE_OUT)
rcf_oil_purification_sys.simulate()

monomer_purification_sys = create_monomer_purification_system(ins=F.PURE_OIL_OUT)
monomer_purification_sys.simulate()

# ── Cellulosic ethanol — Carbohydrate_Pulp feeds directly into fermentation ─
etoh_system = create_cellulosic_ethanol_system(ins=F.Carbohydrate_Pulp)
etoh_system.simulate()

# No pretreatment_wastewater — only S401 stillage filtrate goes to WWT.
etoh_ww     = [F.unit.S401.outs[1]]
etoh_solids = [F.unit.S401.outs[0]]

# ── WWT: RCF wastewater + ethanol stillage filtrate ────────────────────────
WWT = bst.create_conventional_wastewater_treatment_system(
    'WWT',
    ins=[F.RCF_WW_OUTS, F.WW_10, F.WastePulp, F.WW_11, F.WW_12] + etoh_ww,
)
for unit in WWT.units:
    if hasattr(unit, 'strict_moisture_content'):
        unit.strict_moisture_content = False

# Wire WWT RO-treated water to PWC; create_all_facilities(WWT=False) leaves
# M2 (placeholder mixer) empty, so PWC would otherwise buy ~480,000 kg/hr
# of fresh water unnecessarily.
F.unit.PWC.ins[0] = WWT.outs[2]

solids_to_BT = bst.Mixer('MIX_BT_solids', ins=[WWT.outs[1]] + etoh_solids)
gas_mixer    = bst.Mixer('MIX_BT_gas',    ins=[F.RCF_PSAWASTE_OUTS, WWT.outs[0]])

BT = bst.facilities.BoilerTurbogenerator('BT', fuel_price=prices['CH4'])
BT.ins[0] = solids_to_BT.outs[0]
BT.ins[1] = gas_mixer.outs[0]

rcf_pure_mon_etoh_system = bst.System(
    'RCF_PURE_MON_ETOH_system',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, etoh_system, WWT),
    facilities=[solids_to_BT, gas_mixer, BT],
)
rcf_pure_mon_etoh_system.simulate()

integrated_tea = create_cellulosic_ethanol_tea(rcf_pure_mon_etoh_system)

print(f'The MSP for RCF crude oil is  {round(integrated_tea.solve_price(F.MON_MONOMERS_OUT), 3)} USD/kg')



c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.22e+03 hp), head (3.37e+03 ft), kinematic viscosity (6.18e-07 m2/s), and NPSH (3.94 ft); assuming centrigugal pump
  warn(f'{repr(

The MSP for RCF crude oil is  7.047 USD/kg


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: RCF_COMP1> power (0 hp) is out of bounds (10 to 750 hp) for cost correlation
  self._cost(**cost_kwargs) if cost_kwargs else self._cost()


In [2]:

integrated_tea = create_cellulosic_ethanol_tea(rcf_pure_mon_hdo_etoh_etj_system)
mjsp = round(((integrated_tea.solve_price(F.TOTAL_SAF)*F.TOTAL_SAF.rho)/264.172),2)

print(f'The MSP for SAF blend is  {mjsp} USD/gal')


NameError: name 'rcf_pure_mon_hdo_etoh_etj_system' is not defined

In [ ]:
print(f'TCI is {round((integrated_tea.TCI)/1E6,2)} MM USD')

TCI is 1413.45 MM USD


In [ ]:
print(f' Material cost is {round((rcf_pure_mon_hdo_etoh_etj_system.material_cost)/1e6,2)} MM USD/yr')

 Material cost is 350.4 MM USD/yr


In [ ]:
print(f'Utility cost is {round((rcf_pure_mon_hdo_etoh_etj_system.utility_cost)/1e6,2)} MM USD/yr')

Utility cost is 81.04 MM USD/yr


In [ ]:
F.BT.results()

Boiler turbogenerator                                      Units        BT
Electricity           Power                                   kW -2.58e+04
                      Cost                                USD/hr -2.13e+03
Medium pressure steam Duty                                 kJ/hr  -1.1e+07
                      Flow                               kmol/hr      -305
                      Cost                                USD/hr       -84
High pressure steam   Duty                                 kJ/hr -7.98e+08
                      Flow                               kmol/hr -2.48e+04
                      Cost                                USD/hr -7.87e+03
Natural gas           Duty                                 kJ/hr -1.99e+08
                      Flow                               kmol/hr      -295
                      Cost                                USD/hr -1.03e+03
Low pressure steam    Duty                                 kJ/hr -1.09e+09
                      Flow                               kmol/hr -2.81e+04
                      Cost                                USD/hr -6.67e+03
Cooling water         Duty                                 kJ/hr -2.01e+07
                      Flow                               kmol/hr  1.38e+04
                      Cost                                USD/hr      6.71
Fuel (inlet)          Flow                                 kg/hr   2.9e+04
                      Cost                                USD/hr  5.33e+03
Ash disposal (outlet) Flow                                 kg/hr  1.43e+03
                      Cost                                USD/hr      45.5
Design                Work                                    kW  3.17e+04
                      Flow rate                            kg/hr  1.03e+06
                      Ash disposal                         kg/hr  1.43e+03
Purchase cost         Baghouse bags                          USD  2.33e+05
                      Boiler                                 USD  1.05e+08
                      Deaerator                              USD  1.13e+06
                      Amine addition pkg                     USD  1.48e+05
                      Hot process water softener system      USD  2.88e+05
                      Turbogenerator                         USD  1.22e+07
Total purchase cost                                          USD  1.19e+08
Utility cost                                              USD/hr -1.24e+04